In [ ]:
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler
from torch import optim
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
import numpy as np
import os

In [ ]:
data_dir = 'data/images'

batch_size = 32
epochs = 10
workers = 0 if os.name == 'nt' else 8

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Running on device: {}'.format(device))

Running on device: cuda:0


In [ ]:
mtcnn = MTCNN(
    image_size=160, margin=0, min_face_size=20,
    thresholds=[0.6, 0.7, 0.7], factor=0.709, post_process=True,
    device=device
)

In [ ]:
dataset = datasets.ImageFolder(data_dir, transform=transforms.Resize((512, 512)))
dataset.samples = [
    (p, p.replace(data_dir, data_dir + '_cropped'))
        for p, _ in dataset.samples
]
        
loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    collate_fn=training.collate_pil
)

for i, (x, y) in enumerate(loader):
    mtcnn(x, save_path=y)
    print('\rBatch {} of {}'.format(i + 1, len(loader)), end='')
    
# Remove mtcnn to reduce GPU memory usage
del mtcnn


Batch 1 of 2
Batch 2 of 2

In [ ]:
resnet = InceptionResnetV1(
    classify=False,
    pretrained='vggface2',
    num_classes=len(dataset.class_to_idx)
).to(device)

In [ ]:
optimizer = optim.Adam(resnet.parameters(), lr=0.001)
scheduler = MultiStepLR(optimizer, [5, 10])

trans = transforms.Compose([
    np.float32,
    transforms.ToTensor(),
    fixed_image_standardization
])
dataset = datasets.ImageFolder(data_dir + '_cropped', transform=trans)
img_inds = np.arange(len(dataset))
np.random.shuffle(img_inds)
train_inds = img_inds[:int(0.8 * len(img_inds))]
val_inds = img_inds[int(0.8 * len(img_inds)):]

train_loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(train_inds)
)
val_loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(val_inds)
)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
metrics = {
    'fps': training.BatchTimer(),
    'acc': training.accuracy
}

In [ ]:

writer = SummaryWriter()
writer.iteration, writer.interval = 0, 10

print('\n\nInitial')
print('-' * 10)
resnet.eval()
training.pass_epoch(
    resnet, loss_fn, val_loader,
    batch_metrics=metrics, show_running=True, device=device,
    writer=writer
)

for epoch in range(epochs):
    print('\nEpoch {}/{}'.format(epoch + 1, epochs))
    print('-' * 10)

    resnet.train()
    training.pass_epoch(
        resnet, loss_fn, train_loader, optimizer, scheduler,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

    resnet.eval()
    training.pass_epoch(
        resnet, loss_fn, val_loader,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

writer.close()



Initial
----------

Valid |     1/1    | loss:    6.2501 | fps:    2.1024 | acc:    0.0000   

Epoch 1/8
----------

Train |     1/1    | loss:    6.2376 | fps:  118.3510 | acc:    0.0000   

Valid |     1/1    | loss:    6.2996 | fps:   40.5227 | acc:    0.0000   

Epoch 2/8
----------

Train |     1/1    | loss:    6.2332 | fps:  130.1875 | acc:    0.0000   

Valid |     1/1    | loss:    6.1978 | fps:   34.8453 | acc:    0.0000   

Epoch 3/8
----------

Train |     1/1    | loss:    6.2039 | fps:  125.7116 | acc:    0.0000   

Valid |     1/1    | loss:    6.1860 | fps:   39.1778 | acc:    0.0000   

Epoch 4/8
----------

Train |     1/1    | loss:    6.1691 | fps:  133.0033 | acc:    0.1000   

Valid |     1/1    | loss:    6.2015 | fps:   39.3159 | acc:    0.0000   

Epoch 5/8
----------

Train |     1/1    | loss:    6.1625 | fps:  131.3433 | acc:    0.0667   

Valid |     1/1    | loss:    6.2253 | fps:   39.1321 | acc:    0.0000   

Epoch 6/8
----------

Train |     1/1    | 

In [ ]:
resnet.load_state_dict(torch.load('model.pt'))